# Experiment 3 — Round 2: Missing Seeds (Colab Continuation)

This notebook **resumes** Experiment 3 from where Colab left off.  
It only runs the **9 missing method/seed combinations** — everything already in the DB is auto-skipped.

**Missing runs:**
- `FixedTrimmedMean` Seed 3
- `FixedKrum` Seed 3
- `RuleBased` Seed 3
- `SingleShotLLM` Seeds 2 & 3
- `ReflectiveAgent` Seeds 2 & 3
- `AgenticAI` Seeds 2 & 3 (primary novel contribution)

**Setup — do these BEFORE running any cells:**
1. GPU: `Runtime → Change runtime type → T4 GPU`
2. API Key: Left sidebar → Key icon (Secrets) → add `GROQ_API_KEY`
3. Upload these 2 files via the Files panel (folder icon on left):
   - `fl_project_colab.zip`
   - `fl_metrics_experiment3.db` (your DB with 15 completed runs)

**Drive backup:** After each step the DB is auto-saved to your Google Drive root.  
If Colab disconnects mid-run, download `fl_metrics_experiment3.db` from Drive,  
re-upload it here, and re-run from the step that was interrupted — it will resume automatically.

## 0. Mount Google Drive (for auto-backup)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted at /content/drive")
print("DB will be backed up to: /content/drive/MyDrive/fl_metrics_experiment3.db")


## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Install Dependencies

In [ ]:
!pip install -q torch torchvision numpy pydantic groq python-dotenv tabulate matplotlib

## 3. Extract Project & Restore Database

In [ ]:
import os, zipfile, sqlite3, shutil

# 1. Extract the project code
zip_path = '/content/fl_project_colab.zip'
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall('/content')
    print("Extracted fl_project_colab.zip")
else:
    raise FileNotFoundError("fl_project_colab.zip not found! Upload it first.")

# 2. Restore your DB into the project folder
# Priority: uploaded file > Drive backup
db_dst = '/content/fl_project/fl_metrics_experiment3.db'
db_uploaded = '/content/fl_metrics_experiment3.db'
db_drive = '/content/drive/MyDrive/fl_metrics_experiment3.db'

if os.path.exists(db_uploaded):
    shutil.copy2(db_uploaded, db_dst)
    src_label = "uploaded file"
elif os.path.exists(db_drive):
    shutil.copy2(db_drive, db_dst)
    src_label = "Google Drive backup"
else:
    src_label = "fresh DB (no prior runs)"

sz = os.path.getsize(db_dst)
print(f"Restored DB from {src_label}: {sz:,} bytes")

con = sqlite3.connect(db_dst)
cur = con.cursor()
n_runs = cur.execute("SELECT COUNT(*) FROM experiment3_runs").fetchone()[0]
n_rounds = cur.execute("SELECT COUNT(*) FROM experiment3_rounds").fetchone()[0]
con.close()
print(f"  -> {n_runs} runs, {n_rounds} rounds already in DB")


## 4. Configure Environment

In [ ]:
import sys, os
sys.path.insert(0, '/content/fl_project')
os.chdir('/content/fl_project')

from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

import torch
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("GROQ_API_KEY set:", bool(os.environ.get('GROQ_API_KEY')))


## 5. DB Status — What Will Be Run?

In [ ]:
import sqlite3

db_path = '/content/fl_project/fl_metrics_experiment3.db'
con = sqlite3.connect(db_path)
cur = con.cursor()
rows = cur.execute("""
  SELECT runs.method, runs.seed, COUNT(r.id) as rounds_done
  FROM experiment3_rounds r
  JOIN experiment3_runs runs ON r.run_id = runs.run_id
  GROUP BY runs.method, runs.seed
  ORDER BY runs.method, runs.seed
""").fetchall()
con.close()

all_methods = ['AgenticAI','FixedFedAvg','FixedKrum','FixedMedian','FixedTrimmedMean',
               'ReflectiveAgent','RuleBased','SingleShotLLM']
done = {(m, s): n for m, s, n in rows}

print(f"{'Method':<20} | S1       | S2       | S3")
print("-" * 55)
for m in all_methods:
    def fmt(n): return "DONE    " if n==20 else f"{n:>2}/20   "
    s1, s2, s3 = done.get((m,1), 0), done.get((m,2), 0), done.get((m,3), 0)
    print(f"{m:<20} | {fmt(s1)} | {fmt(s2)} | {fmt(s3)}")

print("\nWILL RUN NOW (missing runs):")
missing_count = 0
for m in all_methods:
    for s in [1,2,3]:
        n = done.get((m,s), 0)
        if n < 20:
            missing_count += 1
            print(f"  -> {m} Seed {s}: {n}/20 rounds done")
print(f"\nTotal missing runs: {missing_count}")


---
## Step A — Seed 3 Deterministic Baselines
`FixedTrimmedMean`, `FixedKrum`, `RuleBased` — **no LLM needed**, ~45 min each.  
These are pure computation — no API rate limits apply.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, 'run_experiment3.py',
     '--seeds', '3',
     '--methods', 'FixedTrimmedMean', 'FixedKrum', 'RuleBased',
     '--resume',
     '--no-quota-safe',
     '--db', '/content/fl_project/fl_metrics_experiment3.db',
     '--data-root', '/content/fl_project/data'],
    capture_output=False,
    text=True
)
print("Step A exit code:", result.returncode)


In [ ]:
# ── Auto-backup DB to Google Drive ──
import shutil, os, sqlite3, datetime

db_path = '/content/fl_project/fl_metrics_experiment3.db'
drive_path = '/content/drive/MyDrive/fl_metrics_experiment3.db'

shutil.copy2(db_path, drive_path)
sz = os.path.getsize(drive_path)

con = sqlite3.connect(drive_path)
cur = con.cursor()
n_runs = cur.execute("SELECT COUNT(*) FROM experiment3_runs").fetchone()[0]
n_rounds = cur.execute("SELECT COUNT(*) FROM experiment3_rounds").fetchone()[0]
con.close()

ts = datetime.datetime.now().strftime('%H:%M:%S')
print(f"[{ts}] Drive backup saved: {sz:,} bytes | {n_runs} runs | {n_rounds} rounds")
print(f"  Location: {drive_path}")
print("  To resume after disconnect: re-upload this file from Drive and re-run from the interrupted step.")


In [ ]:
import sqlite3
db_path = '/content/fl_project/fl_metrics_experiment3.db'
con = sqlite3.connect(db_path)
cur = con.cursor()
rows = cur.execute("""
  SELECT runs.method, runs.seed, COUNT(r.id) as n
  FROM experiment3_rounds r
  JOIN experiment3_runs runs ON r.run_id = runs.run_id
  GROUP BY runs.method, runs.seed
""").fetchall()
con.close()
done = {(m,s): n for m,s,n in rows}
print("Step A results:")
for m, s in [('FixedTrimmedMean',3), ('FixedKrum',3), ('RuleBased',3)]:
    n = done.get((m,s), 0)
    print(f"  {m} Seed {s}: {'DONE' if n==20 else f'{n}/20'}")


---
## Step B — SingleShotLLM Seeds 2 & 3
1 LLM call per round × 20 rounds × 2 seeds = **40 API calls total**.  
~15 min, well within free-tier limits.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, 'run_experiment3.py',
     '--seeds', '2', '3',
     '--methods', 'SingleShotLLM',
     '--resume',
     '--db', '/content/fl_project/fl_metrics_experiment3.db',
     '--data-root', '/content/fl_project/data'],
    capture_output=False,
    text=True
)
print("Step B exit code:", result.returncode)


In [ ]:
# ── Auto-backup DB to Google Drive ──
import shutil, os, sqlite3, datetime

db_path = '/content/fl_project/fl_metrics_experiment3.db'
drive_path = '/content/drive/MyDrive/fl_metrics_experiment3.db'

shutil.copy2(db_path, drive_path)
sz = os.path.getsize(drive_path)

con = sqlite3.connect(drive_path)
cur = con.cursor()
n_runs = cur.execute("SELECT COUNT(*) FROM experiment3_runs").fetchone()[0]
n_rounds = cur.execute("SELECT COUNT(*) FROM experiment3_rounds").fetchone()[0]
con.close()

ts = datetime.datetime.now().strftime('%H:%M:%S')
print(f"[{ts}] Drive backup saved: {sz:,} bytes | {n_runs} runs | {n_rounds} rounds")
print(f"  Location: {drive_path}")
print("  To resume after disconnect: re-upload this file from Drive and re-run from the interrupted step.")


---
## Step C — ReflectiveAgent Seeds 2 & 3
~2 LLM calls per round × 20 rounds × 2 seeds = **~80 API calls total**.  
~30 min. Has automatic rate-limit backoff built in.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, 'run_experiment3.py',
     '--seeds', '2', '3',
     '--methods', 'ReflectiveAgent',
     '--resume',
     '--db', '/content/fl_project/fl_metrics_experiment3.db',
     '--data-root', '/content/fl_project/data'],
    capture_output=False,
    text=True
)
print("Step C exit code:", result.returncode)


In [ ]:
# ── Auto-backup DB to Google Drive ──
import shutil, os, sqlite3, datetime

db_path = '/content/fl_project/fl_metrics_experiment3.db'
drive_path = '/content/drive/MyDrive/fl_metrics_experiment3.db'

shutil.copy2(db_path, drive_path)
sz = os.path.getsize(drive_path)

con = sqlite3.connect(drive_path)
cur = con.cursor()
n_runs = cur.execute("SELECT COUNT(*) FROM experiment3_runs").fetchone()[0]
n_rounds = cur.execute("SELECT COUNT(*) FROM experiment3_rounds").fetchone()[0]
con.close()

ts = datetime.datetime.now().strftime('%H:%M:%S')
print(f"[{ts}] Drive backup saved: {sz:,} bytes | {n_runs} runs | {n_rounds} rounds")
print(f"  Location: {drive_path}")
print("  To resume after disconnect: re-upload this file from Drive and re-run from the interrupted step.")


---
## Step D — AgenticAI Seeds 2 & 3 (Primary Contribution)
Multi-agent loop: **Analyst → Proposer → Critic → Repair**.  
~45–60 min per seed. Uses quota-aware rate limiting automatically.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, 'run_experiment3.py',
     '--seeds', '2', '3',
     '--methods', 'AgenticAI',
     '--resume',
     '--db', '/content/fl_project/fl_metrics_experiment3.db',
     '--data-root', '/content/fl_project/data'],
    capture_output=False,
    text=True
)
print("Step D exit code:", result.returncode)


In [ ]:
# ── Auto-backup DB to Google Drive ──
import shutil, os, sqlite3, datetime

db_path = '/content/fl_project/fl_metrics_experiment3.db'
drive_path = '/content/drive/MyDrive/fl_metrics_experiment3.db'

shutil.copy2(db_path, drive_path)
sz = os.path.getsize(drive_path)

con = sqlite3.connect(drive_path)
cur = con.cursor()
n_runs = cur.execute("SELECT COUNT(*) FROM experiment3_runs").fetchone()[0]
n_rounds = cur.execute("SELECT COUNT(*) FROM experiment3_rounds").fetchone()[0]
con.close()

ts = datetime.datetime.now().strftime('%H:%M:%S')
print(f"[{ts}] Drive backup saved: {sz:,} bytes | {n_runs} runs | {n_rounds} rounds")
print(f"  Location: {drive_path}")
print("  To resume after disconnect: re-upload this file from Drive and re-run from the interrupted step.")


---
## Final Audit — All 24 Runs Complete?

In [ ]:
import sqlite3

db_path = '/content/fl_project/fl_metrics_experiment3.db'
con = sqlite3.connect(db_path)
cur = con.cursor()
rows = cur.execute("""
  SELECT runs.method, runs.seed, COUNT(r.id) as n,
         ROUND(AVG(r.test_accuracy),2) as mean_acc,
         ROUND(MAX(r.test_accuracy),2) as max_acc,
         ROUND(MAX(r.cumulative_regret),2) as final_regret
  FROM experiment3_rounds r
  JOIN experiment3_runs runs ON r.run_id = runs.run_id
  GROUP BY runs.method, runs.seed
  ORDER BY runs.method, runs.seed
""").fetchall()
con.close()

all_methods = ['AgenticAI','FixedFedAvg','FixedKrum','FixedMedian','FixedTrimmedMean',
               'ReflectiveAgent','RuleBased','SingleShotLLM']
done = {(m,s): r for m,s,*r in rows}

print(f"{'Method':<20} | S1           | S2           | S3")
print("-" * 65)
for m in all_methods:
    def fmt(s):
        r = done.get((m,s))
        if r is None: return "MISSING  "
        n = r[0]
        return "DONE     " if n==20 else f"{n}/20 !!!"
    print(f"{m:<20} | {fmt(1)} | {fmt(2)} | {fmt(3)}")

total_rounds = sum(r[0] for r in done.values())
total_needed = len(all_methods) * 3 * 20
print(f"\nTotal rounds in DB: {total_rounds} / {total_needed}")
if total_rounds == total_needed:
    print("ALL RUNS COMPLETE! Proceed to results tables.")
else:
    missing = total_needed - total_rounds
    print(f"Still missing {missing} rounds. Re-run incomplete steps above.")


## Results Tables

In [ ]:
import sqlite3
import numpy as np

db_path = '/content/fl_project/fl_metrics_experiment3.db'
con = sqlite3.connect(db_path)
cur = con.cursor()

all_methods = ['AgenticAI','SingleShotLLM','ReflectiveAgent','RuleBased',
               'FixedFedAvg','FixedMedian','FixedTrimmedMean','FixedKrum']

print("=" * 80)
print("TABLE 1: Seed 1 Primary Results")
print("=" * 80)
print(f"{'Method':<20} | {'Final Acc':>9} | {'Max Acc':>7} | {'Mean Acc':>8} | {'Cumul Regret':>12}")
print("-" * 65)
for m in all_methods:
    rows = cur.execute("""
        SELECT r.test_accuracy, r.cumulative_regret
        FROM experiment3_rounds r
        JOIN experiment3_runs runs ON r.run_id = runs.run_id
        WHERE runs.method = ? AND runs.seed = 1
        ORDER BY r.round
    """, (m,)).fetchall()
    if not rows:
        print(f"  {m:<20} | [not in DB]")
        continue
    accs = [r[0] for r in rows]
    print(f"  {m:<20} | {accs[-1]:>8.2f}% | {max(accs):>6.2f}% | {sum(accs)/len(accs):>7.2f}% | {rows[-1][1]:>10.2f}%")

print()
print("=" * 80)
print("TABLE 2: Multi-Seed Mean +/- Std (Seeds 1, 2, 3)")
print("=" * 80)
print(f"{'Method':<20} | {'Mean Regret':>11} | {'Std':>6} | {'Mean Final Acc':>14} | {'Std':>6} | Seeds")
print("-" * 75)
for m in all_methods:
    regrets, final_accs, seeds_done = [], [], []
    for s in [1, 2, 3]:
        row = cur.execute("""
            SELECT r.test_accuracy, r.cumulative_regret
            FROM experiment3_rounds r
            JOIN experiment3_runs runs ON r.run_id = runs.run_id
            WHERE runs.method = ? AND runs.seed = ?
            ORDER BY r.round DESC LIMIT 1
        """, (m, s)).fetchone()
        if row:
            final_accs.append(row[0])
            regrets.append(row[1])
            seeds_done.append(str(s))
    n = len(regrets)
    if n >= 2:
        print(f"  {m:<20} | {np.mean(regrets):>9.2f}%  | {np.std(regrets):>4.2f}% | {np.mean(final_accs):>12.2f}%  | {np.std(final_accs):>4.2f}% | {n}/3")
    elif n == 1:
        print(f"  {m:<20} | {regrets[0]:>9.2f}%  | N/A   | {final_accs[0]:>12.2f}%  | N/A   | 1/3")
    else:
        print(f"  {m:<20} | [no seeds in DB]")

con.close()


---
## Download Updated Database
Run this cell to download the final DB to your PC.  
*(The Drive backup is also always up-to-date from the auto-backup cells above.)*

In [ ]:
from google.colab import files
import shutil, os, sqlite3

db_path = '/content/fl_project/fl_metrics_experiment3.db'
out_path = '/content/fl_metrics_experiment3_updated.db'
shutil.copy2(db_path, out_path)

sz = os.path.getsize(out_path)
con = sqlite3.connect(out_path)
n_rounds = con.execute("SELECT COUNT(*) FROM experiment3_rounds").fetchone()[0]
n_runs   = con.execute("SELECT COUNT(*) FROM experiment3_runs").fetchone()[0]
con.close()

print(f"File: fl_metrics_experiment3_updated.db")
print(f"Size: {sz:,} bytes | Runs: {n_runs} | Rounds: {n_rounds}")
files.download(out_path)
print("Download triggered!")
